In [9]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

current_path = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [current_path, *current_path.parents]
     if (p / "Scripts").is_dir()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the repository.")

# Load data
# FIPS
fips_path = PROJECT_ROOT/"Data/Geography/FIPS/US states FIPS.csv"    # FIPS path
df_fips = pd.read_csv(fips_path)  # Load FIPS
df_fips["FIPS Code"] = df_fips["FIPS Code"].astype(str).str.zfill(2)    # Ensure string, add 0 to single digits

# MSAs
msa_shapefile_path = PROJECT_ROOT/"Data/Geography/CBSA_shapefile_2025/tl_2025_us_cbsa.shp"   # Shapefile path
gdf_msa = gpd.read_file(msa_shapefile_path) # Load MSA shapefile

gdf_msa = gdf_msa[gdf_msa["LSAD"] == 'M1'].reset_index(drop=True)   # Remove micro
gdf_msa['State_Abbr'] = gdf_msa['NAME'].str[-2:]    # Get state abbriviations
gdf_msa = gdf_msa[gdf_msa['State_Abbr'].isin(df_fips['Postal Abbr.'])].copy()   # Filter only US states
valid_geoid = gdf_msa["GEOID"].astype(str).tolist()  # GEOID list

# Lawyers
lawyers_path = PROJECT_ROOT/"Data/BrightData_Lawyers/BrightData_Lawyers_master_normalized_1overN.csv" # Lawyers data Path
lawyers_data = pd.read_csv(lawyers_path)   # Load data
lawyers_data.rename(columns={'CBSA': 'AREA'}, inplace=True) # Rename columns
lawyers_data["AREA"] = lawyers_data["AREA"].astype("Int64").astype(str)
lawyers_data = lawyers_data[lawyers_data["AREA"].isin(valid_geoid)]   # Take only MSAs from the list

# Proxies
# Bankruptcy lawyers
Bankruptcy_path = PROJECT_ROOT/"Data/Proxies/Bankruptcy/bf_f5a_1231.2024_clean.xlsx" # Bankruptcy data Path
Bankruptcy_county_data = pd.read_excel(Bankruptcy_path)   # Load data

Bankruptcy_county_data = Bankruptcy_county_data[['County Code', 'Circuit, District, and County', 'Total All Chapters']].copy()  # Take relevant columns
Bankruptcy_county_data.rename(columns={'Circuit, District, and County': 'County', 'Total All Chapters': 'Bankruptcy filings'}, inplace=True) # Rename columns
Bankruptcy_county_data['County Code'] = Bankruptcy_county_data['County Code'].astype(str).str.rstrip('*')
Bankruptcy_county_data['County Code'] = Bankruptcy_county_data['County Code'].astype(str).str.strip().replace({'nan': pd.NA, 'None': pd.NA, '': pd.NA})
Bankruptcy_county_data = Bankruptcy_county_data.dropna(subset=['County Code']).reset_index(drop=True)

# County to MSA aggregation
crosswalk_path = PROJECT_ROOT/"Data/Geography/Crosswalks/qcew-county-msa-csa-crosswalk-clean.xlsx" # County to MSA crosswalk
crosswalk = pd.read_excel(crosswalk_path)   # Load data

crosswalk['MSA Code'] = crosswalk['MSA Code'].str[-4:].str.ljust(5, '0')    # Convert to MSA GEOID
crosswalk['County Code'] = crosswalk['County Code'].astype(str) # Convert to string
crosswalk['County Code'] = crosswalk['County Code'].astype(str).str.zfill(5)    # Padding

Bankruptcy_county_data = Bankruptcy_county_data.merge(crosswalk[['County Code', 'MSA Code', 'MSA Title']], on='County Code', how='left')    # Add MSA code and name
Bankruptcy_county_data = Bankruptcy_county_data.dropna(subset=['MSA Code']).reset_index(drop=True)  # Drop out-of-MSA counties
Bankruptcy_msa_data = Bankruptcy_county_data.groupby(['MSA Code', 'MSA Title'], as_index=False).agg(Bankruptcy_filings=('Bankruptcy filings', 'sum'))   # Aggregate to MSAs and Micros
Bankruptcy_msa_data.rename(columns={'MSA Code': 'AREA', 'MSA Title': 'MSA', 'Bankruptcy_filings': 'Bankruptcy filings'}, inplace=True) # Rename columns
Bankruptcy_msa_data = Bankruptcy_msa_data[Bankruptcy_msa_data["AREA"].isin(valid_geoid)]   # Take only MSAs from the list
Bankruptcy_msa_data = Bankruptcy_msa_data.merge(lawyers_data[['AREA', 'Bankruptcy Law_normalized_1overN_count']], on=['AREA'], how='left')  # Add lawyers data
Bankruptcy_msa_data = Bankruptcy_msa_data.dropna(subset=['Bankruptcy Law_normalized_1overN_count']) # Drop missing data
Bankruptcy_msa_data = Bankruptcy_msa_data.rename(columns={"Bankruptcy Law_normalized_1overN_count": "Bankruptcy"})
Bankruptcy_msa_data = Bankruptcy_msa_data.drop(columns=["MSA", "MSA_Name", "msa_name"], errors="ignore")

Bankruptcy_msa_data.to_csv(PROJECT_ROOT/"Data/Proxies/Bankruptcy/Bankruptcy_Proxy_Normalized.csv", index=False)
